# RAG Stress-Test — Retriever Ablation (Colab, FIXED)

Runs `faiss_only` and `no_rerank` ablations across all 8 conditions on a GPU.

**Fix vs prior run:** the MedCPT query encoder is forced onto CPU (FP32) so its
embeddings match the Mac-built FAISS index. CUDA kernels produced subtly
different floats that silently missed against the index, giving MAP=0 results.
Ollama (LLaMA) still uses GPU — only the small encoder runs on CPU (~2 sec each).

**Before running:**
1. On your Mac: push the latest code, `git push origin main`
2. Make sure `corpus.db`, `faiss.index`, `questions.json` already live in
   `MyDrive/rag_stress_test_data/` (from prior run, no need to re-upload)
3. Runtime → Change runtime type → T4 GPU → Save
4. Runtime → Run all

## 1. Mount Drive and verify data files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA = '/content/drive/MyDrive/rag_stress_test_data'
for fname in ['corpus.db', 'faiss.index', 'questions.json']:
    p = os.path.join(DRIVE_DATA, fname)
    assert os.path.exists(p), f'MISSING: {p}'
    print(f'OK  {fname}: {os.path.getsize(p)/1e6:.1f} MB')

## 2. Clone the repo (with the encoder-on-CPU fix)

In [ ]:
%cd /content
!rm -rf rag_stress_test
!git clone https://github.com/Tanushree28/rag_stress_test.git
%cd rag_stress_test

## 3. Copy data files from Drive into the repo

In [ ]:
!mkdir -p data
!cp "/content/drive/MyDrive/rag_stress_test_data/corpus.db" data/
!cp "/content/drive/MyDrive/rag_stress_test_data/faiss.index" data/
!cp "/content/drive/MyDrive/rag_stress_test_data/questions.json" data/
# Wipe any prior ablation DB so we start fresh
!rm -f data/experiments_ablation.db
!ls -lh data/

## 4. Install dependencies — note: faiss-CPU (matches Mac-built index)

In [ ]:
!pip install -q faiss-cpu transformers sentence-transformers ollama scipy rouge-score numpy

## 5. Force the encoder onto CPU (the fix)

This single env var causes `retriever.py` to load MedCPT and the cross-encoder
on CPU even though we have a GPU. Ollama (LLaMA generation) still uses GPU.

In [ ]:
import os
os.environ['RAG_FORCE_CPU_ENCODER'] = '1'
print('RAG_FORCE_CPU_ENCODER =', os.environ['RAG_FORCE_CPU_ENCODER'])

## 6. Install + start Ollama, pull LLaMA 3.1:8b (~6 min)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!ollama --version

In [ ]:
!ollama pull llama3.1:8b

## 7. SANITY CHECK — verify FAISS retrieval is working

If the rerank scores below are roughly the same as on Mac
(`hybrid` ~6.5, `faiss_only` ~2.0+, `no_rerank` ~0.6+), the fix worked.
If any are near zero, **STOP** — don't waste 5 hr running broken retrieval.

In [ ]:
import sys; sys.path.insert(0, '.')
from retriever import retrieve

Q = 'What causes Hirschsprung disease?'
for mode in ['hybrid', 'faiss_only', 'bm25_only', 'no_rerank']:
    p = retrieve(Q, top_k=5, mode=mode)
    score = p[0].get('rerank_score') if p else None
    status = 'OK' if (score is not None and score > 0.5) else 'BROKEN'
    print(f'{status:<7} {mode:<12} top rerank={score}')

# Hard assertion -- abort the notebook if any mode is broken
for mode in ['hybrid', 'faiss_only', 'no_rerank']:
    p = retrieve(Q, top_k=5, mode=mode)
    assert p and p[0]['rerank_score'] > 0.4, f'BROKEN: {mode} top score is {p[0]["rerank_score"] if p else None}'
print('\nSANITY CHECK PASSED -- safe to proceed')

## 8. Run the ablation — only the 2 modes that need redoing

`bm25_only` was correct from the prior run; we kept those rows on the Mac.
Now we only re-run `faiss_only` and `no_rerank`.

In [ ]:
!RAG_FORCE_CPU_ENCODER=1 python run_ablation.py --n_per_type 25 --modes faiss_only,no_rerank

## 9. Save results to Drive AND offer direct download

In [ ]:
import shutil, os
src = 'data/experiments_ablation.db'
assert os.path.exists(src), 'No results -- did the run finish?'
shutil.copy(src, '/content/drive/MyDrive/rag_stress_test_data/experiments_ablation_v3.db')
print(f'Copied to Drive: rag_stress_test_data/experiments_ablation_v3.db')
print(f'Size: {os.path.getsize(src)/1e6:.2f} MB')
from google.colab import files
files.download(src)

## On your Mac, after download:

First delete the broken rows, then merge the new ones.

```bash
# 1. Move downloaded file into project
mv ~/Downloads/experiments_ablation.db data/experiments_ablation_v3.db

# 2. Delete broken faiss_only / no_rerank rows from main DB
sqlite3 data/experiments.db \
  "DELETE FROM experiments WHERE retriever_mode IN ('faiss_only','no_rerank');"

# 3. Merge fresh ablation rows in
sqlite3 data/experiments.db \
  "ATTACH 'data/experiments_ablation_v3.db' AS abl; \
   INSERT INTO experiments \
     (timestamp, question_id, question_type, question_body, condition, \
      answer, metrics, passages, duration_s, retriever_mode) \
   SELECT timestamp, question_id, question_type, question_body, condition, \
          answer, metrics, passages, duration_s, retriever_mode \
   FROM abl.experiments;"
```

Then refresh `http://localhost:3000/analytics → Ablation` and the chart will
show real, comparable numbers for all 4 modes.